# Drishti — LoRA fine-tuning on VizWiz (Phase 3)

The project's own ML contribution. Everything before this wires existing models together;
this is the part that produces a result of its own.

## The bar is 0.533, not 0.308

Stock SmolVLM-Instruct scored **0.308** on 500 VizWiz-val samples. Changing the prompt —
no training at all — lifted it to **0.533** (`DEC-016`). Fine-tuning has to beat the number
that is already banked, or it has produced nothing: comparing against 0.308 would credit
training with a gain that prompt engineering already delivered (`DEC-017`).

## What to fine-tune *for*

Not general answering ability. 49% of VizWiz-val is unanswerable — blurry, dark, mis-framed,
the photograph a person who cannot see the framing actually takes. Stock SmolVLM scores
0.306 there because it guesses instead of declining. The arithmetic (`DEC-011`):

| change | overall |
|---|---|
| lift the *unanswerable* subset to 1.0 | 0.308 → 0.647 (**+0.34**) |
| lift *answerable* accuracy to 0.50 | 0.308 → 0.405 (+0.10) |

So the training set over-samples unanswerable examples. That has an obvious failure mode,
and §5 is built to catch it: **a model that abstains on everything scores well on half the
benchmark and is useless.** Abstention precision and recall are reported separately, never
folded into one number (`DEC-014`).

## Everything is held fixed except the weights

The prompt, the 500 evaluation samples, the metric and the normalisation are identical to
notebooks 01 and 02. That is the only way the comparison means anything — a fine-tune
evaluated on a different slice, or without the stakes prompt, is not comparable to 0.533.

Colab: **Runtime → T4 GPU.** Training a LoRA adapter on a ~2B model needs the GPU; the
evaluation pass alone takes about ten minutes on a T4 and hours on CPU.

## 0. Setup

`transformers<5` is pinned for the same reason as everywhere else in this project
(`DEC-009`), and `peft` supplies the LoRA implementation.

In [1]:
%pip install -q "transformers<5" peft accelerate datasets bitsandbytes
# Colab preinstalls torchao 0.10.0; peft requires >0.16 and raises ImportError at
# get_peft_model() rather than at import, so it fails only after the 4.5 GB model has
# downloaded. Nothing here uses torchao -- it is consulted for quantization paths this
# notebook never touches -- and peft's is_torchao_available() returns False cleanly when
# the package is absent. Removing it is lower risk than upgrading it underneath torch.
%pip uninstall -q -y torchao

import json
import os
import re
import string
import subprocess
import sys
import time
from itertools import islice
from pathlib import Path

import os

# Fragmentation, not capacity, is what usually kills this run: the last failure had
# 510 MB reserved-but-unallocated. Must be set before torch initialises CUDA.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import torch
from PIL import Image

REPO_URL = 'https://github.com/DevGurav/Drishti.git'
ON_COLAB = 'google.colab' in sys.modules or Path('/content').is_dir()

if ON_COLAB:
    PROJECT = Path('/content/drishti')
    if not PROJECT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT)], check=True)
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=False)
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

OUT_DIR = PROJECT / 'models' / 'smolvlm-vizwiz-lora'
RESULTS = PROJECT / 'eval' / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'HuggingFaceTB/SmolVLM-Instruct'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if DEVICE == 'cpu':
    print('WARNING: no GPU. Training will not finish in a sensible time, and the '
          'evaluation pass alone takes hours (measured 294s per scene answer on an '
          'i5-11300H). Runtime -> Change runtime type -> T4 GPU.')

print('device:', DEVICE, '| project:', PROJECT)
# Fail before the 4.5 GB download if the environment cannot support LoRA.
import importlib.metadata as _md
try:
    _ao = _md.version('torchao')
    print(f'WARNING: torchao {_ao} is still installed. If get_peft_model() raises an '
          f'ImportError about torchao, restart the runtime -- the uninstall above only '
          f'takes effect on a fresh process.')
except _md.PackageNotFoundError:
    pass

import peft
print('peft:', peft.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
device: cuda | project: /content/drishti
peft: 0.19.1


## 1. The prompt, the metric, the slice — imported, not retyped

Every one of these is copied from the app or from notebook 01. If the suffix here drifted
from `app/engines/smolvlm.py`, the number this notebook produces would describe a model the
app never runs.

In [2]:
# The exact suffix the app ships (DEC-016). Read from the source file rather than
# retyped, so the two cannot silently diverge.
smolvlm_src = (PROJECT / 'app' / 'engines' / 'smolvlm.py').read_text(encoding='utf-8')
match = re.search(r'ABSTENTION_SUFFIX = \(\n(.*?)\n\)', smolvlm_src, re.S)
PROMPT_SUFFIX = ''.join(
    re.findall(r'"([^"]*)"', match.group(1))
) if match else None

assert PROMPT_SUFFIX and 'unanswerable' in PROMPT_SUFFIX, 'could not read ABSTENTION_SUFFIX'
print(repr(PROMPT_SUFFIX))

# --- the VizWiz metric, identical to notebook 01 -------------------------------------
ARTICLES = {'a', 'an', 'the'}


def norm(t):
    t = t.lower().strip()
    t = t.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(w for w in t.split() if w not in ARTICLES)


def vizwiz_acc(pred, answers):
    p = norm(pred)
    return min(sum(norm(a) == p for a in answers) / 3.0, 1.0)


def gt_answers(sample):
    """Answers arrive as list[str] or list[{'answer': ...}] depending on the loader."""
    return [a['answer'] if isinstance(a, dict) else a for a in sample['answers']]


def is_unanswerable(answers):
    """VizWiz convention: unanswerable when most annotators said so."""
    return sum(norm(a) == 'unanswerable' for a in answers) >= 5


def strip_answer_prefix(text):
    """SmolVLM emits 'Answer: unanswerable' intermittently (DEC-018). Unhandled, the app
    reads that string aloud to a blind user instead of offering retake guidance."""
    return re.sub(r'^\s*answer\s*:\s*', '', text.strip(), flags=re.I)

' The person asking is blind and cannot check your answer, so a confident wrong answer is worse than no answer. Answer in one to three words only if the image clearly shows it. Otherwise answer exactly: unanswerable'


## 2. Training data — the official train split, over-sampled for abstention

`lmms-lab/VizWiz-VQA` on Hugging Face publishes **`test` and `val` only** — it is an
evaluation dataset, and `split='train'` raises `ValueError: Bad split`. The 20,523-pair
train split exists solely as an 11.3 GB archive, so the annotations come from
`download_vizwiz.py` (21 MB) and the images are pulled out of the archive individually over
HTTP range requests. A few thousand images, not eleven gigabytes.

Training on `val` was the tempting shortcut and is rejected: evaluation is `val[:500]`, and
fine-tuning on the rest of the same split would flatter the result in a way no reader could
check. Train and eval stay on opposite sides of the official boundary.

`DEC-011` in one line: teaching the model to decline is worth roughly three times more than
teaching it to answer better. `ABSTAIN_RATIO` is the share of training targets that are
`unanswerable` — VizWiz train is about 28%; raising it pushes the decision threshold.

**It is the single most dangerous knob here.** Turn it to 1.0 and §6 will show a model that
abstains on everything, scores about 0.49, and is worthless (`DEC-014`).

In [3]:
sys.path.insert(0, str(PROJECT / 'data' / 'scripts'))
from vizwiz_images import fetch as fetch_vizwiz          # noqa: E402

N_TRAIN = 1500          # at batch 1 this is ~1500 steps; 3000 risks the free-tier
                        # session limit, and the goal is recalibration rather than
                        # teaching new visual skills
ABSTAIN_RATIO = 0.45    # share of 'unanswerable' targets. See the warning above.
SEED = 42

ANNOTATIONS = PROJECT / 'data' / 'vizwiz' / 'train.json'
TRAIN_IMAGES = PROJECT / 'data' / 'vizwiz' / 'images' / 'train'

if not ANNOTATIONS.exists():
    subprocess.run([sys.executable,
                    str(PROJECT / 'data' / 'scripts' / 'download_vizwiz.py')], check=True)

entries = json.loads(ANNOTATIONS.read_text(encoding='utf-8'))
print(f'{len(entries)} train annotations')

# Choose the examples first, then fetch only those images.
answerable, unanswerable = [], []
need_una = int(N_TRAIN * ABSTAIN_RATIO)
need_ans = N_TRAIN - need_una

import random
for entry in random.Random(SEED).sample(entries, len(entries)):
    answers = gt_answers(entry)
    if is_unanswerable(answers):
        if len(unanswerable) < need_una:
            unanswerable.append(entry)
    elif len(answerable) < need_ans:
        answerable.append(entry)
    if len(answerable) >= need_ans and len(unanswerable) >= need_una:
        break

train_entries = answerable + unanswerable
random.Random(SEED).shuffle(train_entries)

print(f'{len(train_entries)} training examples')
print(f'  answerable   : {len(answerable)}')
print(f'  unanswerable : {len(unanswerable)}  '
      f'({len(unanswerable)/max(len(train_entries),1):.0%})')

# Ranged reads out of the 11.3 GB archive: only these images cross the network. Already
# present files are skipped, so an interrupted fetch resumes.
print(f'\nfetching {len(train_entries)} images into {TRAIN_IMAGES}')
tally = fetch_vizwiz('train', [e['image'] for e in train_entries], TRAIN_IMAGES)
print(f"  written {tally['written']} · skipped {tally['skipped']} · "
      f"missing {tally['missing']}")

train_entries = [e for e in train_entries if (TRAIN_IMAGES / e['image']).exists()]
print(f'{len(train_entries)} examples with an image on disk')

20523 train annotations
1500 training examples
  answerable   : 825
  unanswerable : 675  (45%)

fetching 1500 images into /content/drishti/data/vizwiz/images/train
  100 fetched
  200 fetched
  300 fetched
  400 fetched
  500 fetched
  600 fetched
  700 fetched
  800 fetched
  900 fetched
  1000 fetched
  1100 fetched
  1200 fetched
  1300 fetched
  1400 fetched
  1500 fetched
  written 1500 · skipped 0 · missing 0
1500 examples with an image on disk


## 3. Model and LoRA adapter

LoRA rather than full fine-tuning because the whole project targets a phone: an adapter is
a few megabytes on top of weights that are already quantised for the device, and a T4 cannot
full-fine-tune a 2B vision-language model regardless.

The vision tower stays frozen. The failure being fixed is a *decision* failure — the model
sees the blur perfectly well and answers anyway — so the language side is where it lives.

In [4]:
from peft import LoraConfig, get_peft_model
from transformers import AutoProcessor

try:
    from transformers import AutoModelForImageTextToText as _VisionSeq
except ImportError:
    from transformers import AutoModelForVision2Seq as _VisionSeq

# Free anything a previous run left on the card.
#
# This cell OOMed at model.to(DEVICE) -- while merely *moving* a 4.5 GB model -- because
# 14.34 GB was already allocated by the failed attempt before it. A notebook cell that
# cannot be re-run after an error is a trap: the obvious response to a crash is to fix a
# line and press play, and that quietly asks for a second copy of the weights.
import gc

for _name in ('model', 'base_model'):
    if _name in dir():
        del globals()[_name]
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {free/1e9:.1f} GB free of {total/1e9:.1f} GB')
    if free < 9e9:
        raise RuntimeError(
            f'Only {free/1e9:.1f} GB free -- something else is still holding the GPU.\n'
            'Runtime -> Restart session, then Run all. Clearing Python references cannot '
            'release memory held by a process that is still alive.'
        )

processor = AutoProcessor.from_pretrained(MODEL_ID)

# device_map streams the weights straight onto the GPU shard by shard. Loading to CPU and
# calling .to() afterwards briefly needs room for both copies, which is what failed.
model = _VisionSeq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    low_cpu_mem_usage=True,
    device_map={'': 0} if DEVICE == 'cuda' else None,
)
if DEVICE != 'cuda':
    model.to(DEVICE)

# Attention projections on the language side only.
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

model = get_peft_model(model, lora)

# Recompute activations instead of storing them. SmolVLM tiles each image into
# sub-images, so one sample is well over a thousand visual tokens and the stored
# activations -- not the 4.5 GB of weights -- are what exhausted a 14.5 GB T4.
model.gradient_checkpointing_enable()
model.enable_input_require_grads()   # required when the embeddings are frozen

# The KV cache exists to speed up generation and is pure overhead while training. Left on
# it also fights gradient checkpointing, which recomputes the very activations it caches.
model.config.use_cache = False

# The base stays fp16; the trainable LoRA parameters go to fp32. Adam on fp16 master
# weights underflows on updates this small, which shows up as a loss that never moves.
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

model.print_trainable_parameters()

GPU: 15.5 GB free of 15.6 GB


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/4.49G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

trainable params: 9,277,440 || all params: 2,255,550,320 || trainable%: 0.4113


## 4. Train

Short and deliberately unambitious. The aim is to move the abstention threshold, not to
teach the model new visual skills — and a longer run on 3,000 examples mostly buys
overfitting to VizWiz's phrasing.

In [5]:
from torch.utils.data import DataLoader, Dataset

EPOCHS = 1
BATCH = 1            # a T4 cannot hold two tiled SmolVLM samples at once
ACCUM = 8            # so the effective batch is 8, via accumulation
LR = 1e-4


def target_answer(sample):
    answers = gt_answers(sample)
    if is_unanswerable(answers):
        return 'unanswerable'
    normed = [norm(a) for a in answers]
    return max(set(normed), key=normed.count)


class VizWizSFT(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        s = self.samples[i]
        msgs = [{'role': 'user',
                 'content': [{'type': 'image'},
                             {'type': 'text', 'text': s['question'] + PROMPT_SUFFIX}]}]
        prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
        # Train images come from the archive as files; val images arrive from the HF
        # dataset as PIL objects. Handle both so the two paths share this class.
        image = s['image']
        if isinstance(image, str):
            image = Image.open(TRAIN_IMAGES / image)
        return {'image': image.convert('RGB'),
                'prompt': prompt,
                'target': target_answer(s)}


def collate(batch):
    texts = [b['prompt'] + b['target'] + processor.tokenizer.eos_token for b in batch]
    images = [[b['image']] for b in batch]
    enc = processor(text=texts, images=images, return_tensors='pt', padding=True)
    labels = enc['input_ids'].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    enc['labels'] = labels
    return enc


loader = DataLoader(VizWizSFT(train_entries), batch_size=BATCH, shuffle=True,
                    collate_fn=collate)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)

# fp16 autocast with a loss scaler. The T4 is Turing, which has no bfloat16, so
# plain fp16 it is -- and unscaled fp16 gradients on LoRA-sized updates underflow to
# zero, which looks exactly like a model that refuses to learn.
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

model.train()
for epoch in range(1, EPOCHS + 1):
    running, seen, t0 = 0.0, 0, time.time()
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.amp.autocast('cuda', dtype=torch.float16, enabled=(DEVICE == 'cuda')):
            loss = model(**batch).loss / ACCUM

        scaler.scale(loss).backward()

        if step % ACCUM == 0 or step == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running += loss.item() * ACCUM * batch['input_ids'].size(0)
        seen += batch['input_ids'].size(0)

        if step == 1 or step % 200 == 0:
            used = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == 'cuda' else 0
            print(f'  epoch {epoch} step {step}/{len(loader)}  loss {running/seen:.4f}'
                  f'  peak {used:.1f} GB', flush=True)

    print(f'epoch {epoch}: loss {running/seen:.4f}  ({time.time()-t0:.0f}s)')

model.eval()
OUT_DIR.parent.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUT_DIR)
print(f'\nadapter saved to {OUT_DIR}')

  epoch 1 step 1/1500  loss 17.9095  peak 6.1 GB
  epoch 1 step 200/1500  loss 11.4314  peak 6.3 GB
  epoch 1 step 400/1500  loss 6.0398  peak 6.3 GB
  epoch 1 step 600/1500  loss 4.1093  peak 6.7 GB
  epoch 1 step 800/1500  loss 3.1331  peak 6.7 GB
  epoch 1 step 1000/1500  loss 2.5406  peak 6.7 GB
  epoch 1 step 1200/1500  loss 2.1395  peak 6.7 GB
  epoch 1 step 1400/1500  loss 1.8479  peak 6.7 GB
epoch 1: loss 1.7303  (5982s)

adapter saved to /content/drishti/models/smolvlm-vizwiz-lora


## 5. Evaluate — same 500 samples, same prompt, same metric

`islice(stream, 500)` over `val` is exactly what notebooks 01 and 02 evaluated. Nothing
about the measurement changes; only the weights do.

In [6]:
from datasets import load_dataset

N_SAMPLES = 500

val = list(islice(load_dataset('lmms-lab/VizWiz-VQA', split='val', streaming=True),
                  N_SAMPLES))
print(f'{len(val)} evaluation samples')


@torch.no_grad()
def answer(image, question):
    msgs = [{'role': 'user',
             'content': [{'type': 'image'},
                         {'type': 'text', 'text': question + PROMPT_SUFFIX}]}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image.convert('RGB')],
                       return_tensors='pt').to(DEVICE)
    out = model.generate(**inputs, max_new_tokens=20, do_sample=False)
    text = processor.batch_decode(out[:, inputs['input_ids'].shape[1]:],
                                  skip_special_tokens=True)[0]
    return strip_answer_prefix(text)


results = []
t0 = time.time()
for i, sample in enumerate(val, 1):
    started = time.time()
    prediction = answer(sample['image'], sample['question'])
    answers = gt_answers(sample)
    results.append({
        'question': sample['question'],
        'prediction': prediction,
        'answers': answers,
        'acc': vizwiz_acc(prediction, answers),
        'unanswerable_gt': is_unanswerable(answers),
        'said_unanswerable': norm(prediction) == 'unanswerable',
        'latency_s': time.time() - started,
    })
    if i % 50 == 0:
        print(f'  {i}/{len(val)}  ({time.time()-t0:.0f}s)', flush=True)

print(f'done in {time.time()-t0:.0f}s')

README.md: 0.00B [00:00, ?B/s]

500 evaluation samples
  50/500  (77s)
  100/500  (153s)
  150/500  (228s)
  200/500  (303s)
  250/500  (376s)
  300/500  (449s)
  350/500  (522s)
  400/500  (601s)
  450/500  (673s)
  500/500  (746s)
done in 746s


## 6. The numbers that decide whether this worked

Aggregate accuracy alone cannot answer that. A model that abstains on everything scores
about 0.49 here and is useless, so abstention precision and recall are reported separately
(`DEC-014`):

- **recall** — of the questions that genuinely cannot be answered, how many did it decline?
- **precision** — when it declined, how often was it right to?

The stock model had precision 0.913 and recall 0.258: it was almost always right to abstain
and almost never did. The stakes prompt moved recall to 0.639 at a precision cost. Training
should raise **both**, which prompting alone could not — every prompt variant traded one for
the other.

In [7]:
overall = sum(r['acc'] for r in results) / len(results)
ans = [r for r in results if not r['unanswerable_gt']]
una = [r for r in results if r['unanswerable_gt']]

tp = sum(1 for r in results if r['said_unanswerable'] and r['unanswerable_gt'])
fp = sum(1 for r in results if r['said_unanswerable'] and not r['unanswerable_gt'])
fn = sum(1 for r in results if not r['said_unanswerable'] and r['unanswerable_gt'])
precision = tp / (tp + fp) if tp + fp else float('nan')
recall = tp / (tp + fn) if tp + fn else float('nan')

print(f'FINE-TUNED — {len(results)} VizWiz-val samples, stakes prompt')
print(f'  overall accuracy    : {overall:.3f}')
print(f'  answerable subset   : {sum(r["acc"] for r in ans)/max(len(ans),1):.3f}  (n={len(ans)})')
print(f'  unanswerable subset : {sum(r["acc"] for r in una)/max(len(una),1):.3f}  (n={len(una)})')
print(f'  abstention precision: {precision:.3f}')
print(f'  abstention recall   : {recall:.3f}')
print(f'  mean latency        : {sum(r["latency_s"] for r in results)/len(results):.2f}s')

# The degenerate solution DEC-014 exists to expose.
abstain_rate = sum(r['said_unanswerable'] for r in results) / len(results)
print(f'\n  said "unanswerable" on {abstain_rate:.0%} of all questions')
if abstain_rate > 0.75:
    print('  *** This is close to abstaining on everything. A model that always declines')
    print('      scores well on the unanswerable half and helps nobody. Lower')
    print('      ABSTAIN_RATIO and retrain before reporting this number. ***')

print()
print(f"{'':22}{'stock':>9}{'stakes':>9}{'tuned':>9}")
print('-' * 49)
print(f"{'overall':22}{0.308:>9.3f}{0.533:>9.3f}{overall:>9.3f}")
print(f"{'unanswerable subset':22}{0.306:>9.3f}{0.673:>9.3f}"
      f"{sum(r['acc'] for r in una)/max(len(una),1):>9.3f}")
print(f"{'abstention precision':22}{0.913:>9.3f}{0.726:>9.3f}{precision:>9.3f}")
print(f"{'abstention recall':22}{0.258:>9.3f}{0.639:>9.3f}{recall:>9.3f}")
print()
verdict = ('BEATS the 0.533 prompt-only result' if overall > 0.533
           else 'does NOT beat 0.533 -- prompting alone still wins')
print(f'verdict: fine-tuning {verdict}')

FINE-TUNED — 500 VizWiz-val samples, stakes prompt
  overall accuracy    : 0.521
  answerable subset   : 0.370  (n=256)
  unanswerable subset : 0.680  (n=244)
  abstention precision: 0.581
  abstention recall   : 0.664
  mean latency        : 1.49s

  said "unanswerable" on 56% of all questions

                          stock   stakes    tuned
-------------------------------------------------
overall                   0.308    0.533    0.521
unanswerable subset       0.306    0.673    0.680
abstention precision      0.913    0.726    0.581
abstention recall         0.258    0.639    0.664

verdict: fine-tuning does NOT beat 0.533 -- prompting alone still wins


## 7. Save the results and the adapter

`eval/results/*.csv` is versioned deliberately (`DEC-015`): every claim in the writeup should
trace to a file, and they are tens of kilobytes.

The adapter is a few megabytes and lives in `models/`, which is gitignored — download it,
exactly as with the currency checkpoint. A training run whose output dies with the Colab VM
has produced a number and nothing else.

In [8]:
import csv

csv_path = RESULTS / 'vizwiz_lora_results.csv'
with csv_path.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.writer(fh)
    writer.writerow(['question', 'prediction', 'answers', 'acc', 'unanswerable_gt',
                     'said_unanswerable', 'latency_s'])
    for r in results:
        writer.writerow([r['question'], r['prediction'], '|'.join(r['answers']),
                         f"{r['acc']:.3f}", int(r['unanswerable_gt']),
                         int(r['said_unanswerable']), f"{r['latency_s']:.2f}"])
print(f'wrote {csv_path}')

if ON_COLAB:
    import shutil
    archive = shutil.make_archive(str(PROJECT / 'smolvlm-vizwiz-lora'), 'zip', OUT_DIR)
    print(f'adapter archived: {archive}')
    try:
        from google.colab import files
        files.download(archive)
        files.download(str(csv_path))
    except Exception as e:
        print(f'Browser download unavailable ({type(e).__name__}). Copy to Drive instead:')
        print("    from google.colab import drive; drive.mount('/content/drive')")
        print(f"    import shutil; shutil.copy('{archive}', '/content/drive/MyDrive/')")

wrote /content/drishti/eval/results/vizwiz_lora_results.csv
adapter archived: /content/drishti/smolvlm-vizwiz-lora.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Record for the writeup

1. **The ablation table from §6**, all four rows. Overall accuracy alone hides whether the
   gain came from answering better or declining more, and those are different claims.
2. **Whether both abstention precision and recall rose.** Prompting could only trade one
   for the other; if training raised both, that is the result worth reporting, and if it
   did not, say so — a negative result measured honestly is still a finding (`RISK-5`).
3. **The abstention rate.** If the model declines on most questions it has found the
   degenerate solution, and the overall number is meaningless.
4. **What it cost**: adapter size, training time, and the fact that it runs on a free T4.

If fine-tuning does not beat 0.533, that is publishable and should be stated plainly. The
prompt result is already a genuine finding; a fine-tune that fails to improve on it says
something real about how much of this problem is calibration rather than capability.